# **Prepare dataset images for Cellpose batch processing**

Use this notebook **after** tuning settings in `NEW_infer_masks_cellpose.ipynb`.
It reads raw images from dataset folders, prepares Cellpose-ready TIFF files, and writes batch command files.

This notebook does **not** run Cellpose itself. It only prepares inputs and commands.

### Recommended organization scheme
Inside each dataset folder (example: `KK-1`):
- raw image files remain at the top level
- one analysis subfolder (example: `20261112_batch-test`) contains everything for this Cellpose workflow

For multi-dataset runs, each dataset gets its **own** analysis subfolder inside that dataset folder (same folder name), so datasets are isolated exactly as if run separately.

Inside each analysis subfolder, this notebook writes:
- `cellpose_config.json` and optional `cellpose_config_history/`
- `cell_input/`, `nuclei_input/`, `metadata/`, `cell_masks/`, `nuclei_masks/`
- `prepared_images.csv`
- `run_cellpose_batch.bat`

### Recommended user workflow
1. Tune one dataset in `NEW_infer_masks_cellpose.ipynb`.
2. Save config into the analysis subfolder for that dataset.
3. Run this notebook to prepare files and generate commands.
4. Run each generated command file in a terminal with Cellpose 3 (`cellpose<4`).

### Why this is beginner-friendly
- edit only cells marked **User Input Required**
- run cells top to bottom
- read warning messages; they describe what to fix

---------------------
## **STEP 1: Settings**

#### 🛑 ✍ **User Input Required:**

Choose one settings mode:

1. `SETTINGS_SOURCE = "config"` (**recommended**)
   - Read `cellpose_config.json` from each dataset **workflow output folder**.
   - Best for reproducibility and team use.

2. `SETTINGS_SOURCE = "manual"`
   - Use values in the manual settings block below.
   - Use this mode only when you intentionally want manual values.

Default behavior is strict: this notebook does **not** silently fall back to manual settings when config files are missing.

### Quick start patterns

1. **Config-first (production, strict default)**
   - `SETTINGS_SOURCE = "config"`
   - `STRICT_CONFIG = True`
   - `FALLBACK_TO_MANUAL_IF_MISSING_CONFIG = False`
   - `SHARED_CONFIG_PATH = None`
   - Result: each dataset uses its own existing workflow config.

2. **Manual mode (intentional)**
   - `SETTINGS_SOURCE = "manual"`
   - Fill manual settings in the next cell
   - `WRITE_CONFIG_HISTORY = True`
   - Result: each dataset workflow folder gets an updated config + history snapshot.

3. **Use one shared config for all datasets**
   - `SETTINGS_SOURCE = "config"`
   - `SHARED_CONFIG_PATH = Path("/path/to/existing/cellpose_config.json")`
   - Result: the shared config is applied to every dataset workflow folder before prep runs.

#### ▶ Run this cell first (no user input required)

This imports all required packages for the notebook.
Do not edit this cell.

In [7]:
from datetime import datetime
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile

from infer_subc.core.file_io import list_image_files, read_czi_image
from infer_subc.workflow.cellpose_notebook_utils import (
    REQUIRED_CONFIG_KEYS,
    resolve_dataset_settings,
    update_config_im_type,
    validate_and_normalize_settings,
    normalize_volume_01,
    downsample_zyx,
    build_cell_signal,
    build_cellpose_cli,
)

#### 🛑 Edit the next cell (User Input Required)

Set paths and settings in the next code cell before running anything below it.

Definitions for each user-input variable:

**Dataset + output paths**
- `SOURCE_FOLDERS`: list of dataset folders to process.
- `WORKFLOW_OUTPUT_FOLDER_NAME`: analysis subfolder name created inside each dataset folder (single and multi-dataset).

**Settings source policy**
- `SETTINGS_SOURCE`: `"config"` to read per-dataset config files, or `"manual"` to use manual values below.
- `CONFIG_FILENAME`: config file name to read/write (usually `cellpose_config.json`).
- `STRICT_CONFIG`: when in config mode, require valid config presence.
- `FALLBACK_TO_MANUAL_IF_MISSING_CONFIG`: if `True`, missing configs can fall back to manual settings.
- `SHARED_CONFIG_PATH`: optional shared config path. If set (not `None`) in config mode, that config is applied to all dataset workflow folders before preparation.

**Image-type detection behavior**
- `AUTO_DETECT_IM_TYPE_IF_EMPTY`: if configured `IM_TYPE` has no matches, try common extensions.
- `AUTO_DETECT_IM_TYPE_CANDIDATES`: extension search order for auto-detect.
- `UPDATE_CONFIG_IM_TYPE_ON_AUTODETECT`: persist detected extension back to dataset config.

**Config metadata writing**
- `WRITE_CONFIG_HISTORY`: write timestamped config snapshots.
- `HISTORY_DIRNAME`: history folder name.
- `CONFIG_SELECTED_BY`: metadata user/source field for config writes.
- `CONFIG_NOTES`: free-text metadata notes for config writes.

**Preparation behavior**
- `SKIP_EXISTING`: if `True`, skip images that are already prepared.

**Manual settings block (used only when `SETTINGS_SOURCE="manual"`)**
- `IM_TYPE`: file extension to scan in each dataset.
- `SEGMENT_CELLS`, `SEGMENT_NUCLEI`: enable/disable cell and nuclei preparation/commands.
- `CELL_SOURCE`: `"single"` or `"composite"` cell-signal mode.
- `PM_CH`: plasma membrane/main cell-signal channel index.
- `COMPOSITE_WEIGHTS`: per-channel weights for composite mode.
- `INVERT_PM`: include inverted PM contribution in composite mode.
- `NUC_CH`: nuclei channel index.
- `USE_NUCLEI_FOR_CELLS`: include nuclei channel in cell model input.
- `XY_DOWNSAMPLE`: XY shrink factor before writing prepared TIFFs.
- `CELL_DIAMETER`, `NUC_DIAMETER`: expected diameters used in generated Cellpose commands.
- `FLOW_THRESHOLD`, `CELLPROB_THRESHOLD`: Cellpose thresholds for command generation.
- `MIN_SIZE_CELL`, `MIN_SIZE_NUC`: minimum object sizes for generated commands.
- `BATCH_SIZE`: Cellpose batch size in generated commands.
- `USE_GPU`: include `--use_gpu` in commands.
- `CP_CELL_MODEL_PATH`, `CP_NUC_MODEL_PATH`: optional custom model paths (`None` uses pretrained models).
- `PROFILE_UPSERT_KEYS`: provenance log upsert keys kept in config.
- `DEFAULT_SETTINGS`: fallback dictionary used to fill missing keys in loaded configs.

In [2]:
#### USER INPUT REQUIRED ####
# Point to dataset folders (folder name should be the unique dataset ID).
SOURCE_FOLDERS = [
    Path(r"/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1"),
    # Path(r"/users/s/r/srhoads/MJFF_Cellpose-batch-testing/000001"),
]

# Folder name used when writing outputs inside each dataset folder.
# Resulting path per dataset: <dataset_folder>/<WORKFLOW_OUTPUT_FOLDER_NAME>
WORKFLOW_OUTPUT_FOLDER_NAME = "20261112_batch-test"



# ---------- settings source ----------
SETTINGS_SOURCE = "config"   # "config" or "manual"
CONFIG_FILENAME = "cellpose_config.json"
STRICT_CONFIG = True          # only used when SETTINGS_SOURCE == "config"
FALLBACK_TO_MANUAL_IF_MISSING_CONFIG = False  # strict by default; manual mode should be explicit

# Optional: set one shared config to apply to all datasets in this run (or leave None).
SHARED_CONFIG_PATH = None  # e.g. Path('/.../KK-1/20261112_batch-test/cellpose_config.json')

# If config IM_TYPE has no files, auto-detect from common types in this order.
AUTO_DETECT_IM_TYPE_IF_EMPTY = True
AUTO_DETECT_IM_TYPE_CANDIDATES = [".tiff", ".tif", ".czi"]
UPDATE_CONFIG_IM_TYPE_ON_AUTODETECT = True

# ---------- config writing ----------
WRITE_CONFIG_HISTORY = True
HISTORY_DIRNAME = "cellpose_config_history"
CONFIG_SELECTED_BY = "prep_notebook"
CONFIG_NOTES = ""

SKIP_EXISTING = True



# ---------- manual settings (legacy-style inputs; used only when SETTINGS_SOURCE == "manual") ----------
IM_TYPE = ".tiff"

SEGMENT_CELLS = True
SEGMENT_NUCLEI = True

# --- cell signal ---
CELL_SOURCE = "composite"      # "composite" or "single"
PM_CH = 7                      # plasma membrane channel
COMPOSITE_WEIGHTS = {0: 1,
                     1: 2,
                     2: 3,
                     3: 3,
                     4: 2,
                     5: 2,
                     6: 3}     # channel: weight (used if CELL_SOURCE = "composite")
INVERT_PM = False              # composite only: also add the inverted plasma membrane channel

# --- nuclei ---
NUC_CH = 1                     # nuclear stain channel
USE_NUCLEI_FOR_CELLS = True    # True: 2-channel cell file (cells + nuclei, shared with the nuclei model)

# --- image size ---
XY_DOWNSAMPLE = 1              # >1 shrinks XY (faster); masks are then at the shrunken size

# --- Cellpose parameters (diameters in pixels of the prepared image; 0 = estimate automatically) ---
CELL_DIAMETER = 500
NUC_DIAMETER = 150
FLOW_THRESHOLD = 0.2
CELLPROB_THRESHOLD = -3
MIN_SIZE_CELL = 5000
MIN_SIZE_NUC = 2000
BATCH_SIZE = 4
USE_GPU = True

# Optional custom models (None = pretrained cyto3/nuclei)
CP_CELL_MODEL_PATH = None
CP_NUC_MODEL_PATH = None

# Resource log upsert behavior from NEW_infer_masks_cellpose.ipynb
# (used for provenance in config files; default keeps one latest benchmark row per dataset)
PROFILE_UPSERT_KEYS = ["dataset_id"]

# Fallback defaults used to fill missing keys in configs
DEFAULT_SETTINGS = {
    "IM_TYPE": IM_TYPE,
    "SEGMENT_CELLS": SEGMENT_CELLS,
    "SEGMENT_NUCLEI": SEGMENT_NUCLEI,
    "CELL_SOURCE": CELL_SOURCE,
    "PM_CH": PM_CH,
    "COMPOSITE_WEIGHTS": COMPOSITE_WEIGHTS,
    "INVERT_PM": INVERT_PM,
    "NUC_CH": NUC_CH,
    "USE_NUCLEI_FOR_CELLS": USE_NUCLEI_FOR_CELLS,
    "XY_DOWNSAMPLE": XY_DOWNSAMPLE,
    "CELL_DIAMETER": CELL_DIAMETER,
    "NUC_DIAMETER": NUC_DIAMETER,
    "FLOW_THRESHOLD": FLOW_THRESHOLD,
    "CELLPROB_THRESHOLD": CELLPROB_THRESHOLD,
    "MIN_SIZE_CELL": MIN_SIZE_CELL,
    "MIN_SIZE_NUC": MIN_SIZE_NUC,
    "BATCH_SIZE": BATCH_SIZE,
    "USE_GPU": USE_GPU,
    "CP_CELL_MODEL_PATH": CP_CELL_MODEL_PATH,
    "CP_NUC_MODEL_PATH": CP_NUC_MODEL_PATH,
    "PROFILE_UPSERT_KEYS": PROFILE_UPSERT_KEYS,
}

#### ▶ Run this cell (no user input required)

This loads helper utilities and validates settings early.
If an error appears here, fix the setting named in the error message before continuing.

In [3]:
if not isinstance(SOURCE_FOLDERS, (list, tuple)):
    raise ValueError("SOURCE_FOLDERS must be a list, e.g. [Path('/path/to/dataset')].")

MANUAL_SETTINGS = {
    "IM_TYPE": IM_TYPE,
    "SEGMENT_CELLS": SEGMENT_CELLS,
    "SEGMENT_NUCLEI": SEGMENT_NUCLEI,
    "CELL_SOURCE": CELL_SOURCE,
    "PM_CH": PM_CH,
    "COMPOSITE_WEIGHTS": COMPOSITE_WEIGHTS,
    "INVERT_PM": INVERT_PM,
    "NUC_CH": NUC_CH,
    "USE_NUCLEI_FOR_CELLS": USE_NUCLEI_FOR_CELLS,
    "XY_DOWNSAMPLE": XY_DOWNSAMPLE,
    "CELL_DIAMETER": CELL_DIAMETER,
    "NUC_DIAMETER": NUC_DIAMETER,
    "FLOW_THRESHOLD": FLOW_THRESHOLD,
    "CELLPROB_THRESHOLD": CELLPROB_THRESHOLD,
    "MIN_SIZE_CELL": MIN_SIZE_CELL,
    "MIN_SIZE_NUC": MIN_SIZE_NUC,
    "BATCH_SIZE": BATCH_SIZE,
    "USE_GPU": USE_GPU,
    "CP_CELL_MODEL_PATH": CP_CELL_MODEL_PATH,
    "CP_NUC_MODEL_PATH": CP_NUC_MODEL_PATH,
    "PROFILE_UPSERT_KEYS": PROFILE_UPSERT_KEYS,
}

# Validate once up-front so users get immediate feedback on invalid manual values.
MANUAL_SETTINGS = validate_and_normalize_settings(
    MANUAL_SETTINGS,
    default_settings=DEFAULT_SETTINGS,
    required_keys=REQUIRED_CONFIG_KEYS,
    source_label="manual settings block",
)

if SHARED_CONFIG_PATH is not None:
    SHARED_CONFIG_PATH = Path(SHARED_CONFIG_PATH)
    if not SHARED_CONFIG_PATH.exists():
        raise FileNotFoundError(f"Shared config file not found: {SHARED_CONFIG_PATH}")
else:
    SHARED_CONFIG_PATH = None

print(f"Settings source: {SETTINGS_SOURCE}")
print(f"Config filename: {CONFIG_FILENAME}")
print(f"Fallback to manual when config missing: {FALLBACK_TO_MANUAL_IF_MISSING_CONFIG}")
print(f"Use shared config for all datasets: {SHARED_CONFIG_PATH is not None}")
if SHARED_CONFIG_PATH is not None:
    print(f"Shared config template: {SHARED_CONFIG_PATH}")

Settings source: config
Config filename: cellpose_config.json
Fallback to manual when config missing: False
Use shared config for all datasets: False


---------------------
## **STEP 2: Find images and resolve config locations**

#### ▶ Run this cell (no user input required)

This step:
- scans each dataset folder for raw image files
- resolves settings from config/manual mode
- chooses a workflow output folder per dataset
- reads/writes `cellpose_config.json` in that dataset-specific workflow folder
- builds a table of usable datasets and files

Per-dataset folder naming uses `WORKFLOW_OUTPUT_FOLDER_NAME` for both single and multi-dataset runs.

In [4]:
datasets, used_names = [], set()

for folder in SOURCE_FOLDERS:
    folder = Path(folder)
    if not folder.exists():
        print(f"WARNING: folder not found, skipped: {folder}")
        continue

    workflow_root = folder / WORKFLOW_OUTPUT_FOLDER_NAME
    workflow_root.mkdir(parents=True, exist_ok=True)

    cfg_file = workflow_root / CONFIG_FILENAME
    if SETTINGS_SOURCE == "config" and SHARED_CONFIG_PATH is not None:
        raw = json.loads(Path(SHARED_CONFIG_PATH).read_text())
        if isinstance(raw, dict) and "settings" in raw and isinstance(raw["settings"], dict):
            template_settings = raw["settings"]
        elif isinstance(raw, dict):
            template_settings = raw
        else:
            raise ValueError(f"Shared config content must be a JSON object: {SHARED_CONFIG_PATH}")

        template_settings = validate_and_normalize_settings(
            template_settings,
            default_settings=DEFAULT_SETTINGS,
            required_keys=REQUIRED_CONFIG_KEYS,
            source_label=f"shared config template ({SHARED_CONFIG_PATH})",
        )

        template_payload = {
            "dataset_id": folder.name,
            "config_version": "1.0",
            "selected_on": datetime.now().isoformat(timespec="seconds"),
            "selected_by": CONFIG_SELECTED_BY,
            "notes": f"Applied shared template: {SHARED_CONFIG_PATH}",
            "generated_by": "NEW_cellpose-prep.ipynb",
            "generation_reason": "shared-config-template-all",
            "settings": template_settings,
        }
        cfg_file.write_text(json.dumps(template_payload, indent=2))

        if WRITE_CONFIG_HISTORY:
            hist_dir = workflow_root / HISTORY_DIRNAME
            hist_dir.mkdir(parents=True, exist_ok=True)
            stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            hist_file = hist_dir / f"cellpose_config_{stamp}.json"
            hist_file.write_text(json.dumps(template_payload, indent=2))

        print(f"Applied shared dataset config template: {cfg_file}")

    try:
        cfg, cfg_path, cfg_source = resolve_dataset_settings(
            dataset_folder=folder,
            config_folder=workflow_root,
            settings_source=SETTINGS_SOURCE,
            config_filename=CONFIG_FILENAME,
            strict_config=STRICT_CONFIG,
            fallback_to_manual_if_missing_config=FALLBACK_TO_MANUAL_IF_MISSING_CONFIG,
            default_settings=DEFAULT_SETTINGS,
            manual_settings=MANUAL_SETTINGS,
            selected_by=CONFIG_SELECTED_BY,
            notes=CONFIG_NOTES,
            write_config_history=WRITE_CONFIG_HISTORY,
            history_dirname=HISTORY_DIRNAME,
            generated_by="NEW_cellpose-prep.ipynb",
        )
    except Exception as e:
        print(f"WARNING: config error for {folder}: {type(e).__name__}: {e}")
        continue

    ds_im_type = str(cfg.get("IM_TYPE", DEFAULT_SETTINGS["IM_TYPE"]))
    files = list_image_files(folder, ds_im_type)

    # Optional auto-detect fallback if configured IM_TYPE does not match files in the folder.
    if (not files) and AUTO_DETECT_IM_TYPE_IF_EMPTY:
        detected = None
        for cand in AUTO_DETECT_IM_TYPE_CANDIDATES:
            cand = str(cand)
            cand_files = list_image_files(folder, cand)
            if cand_files:
                detected = cand
                files = cand_files
                break

        if detected is not None:
            print(f"WARNING: no '{ds_im_type}' files in {folder}. Auto-detected '{detected}' ({len(files)} files).")
            ds_im_type = detected
            cfg["IM_TYPE"] = detected
            if UPDATE_CONFIG_IM_TYPE_ON_AUTODETECT and cfg_path is not None:
                update_config_im_type(cfg_path, detected)
                print(f"Updated config IM_TYPE in {cfg_path} -> '{detected}'")

    if not files:
        print(f"WARNING: no '{ds_im_type}' files found in {folder}")
        continue

    name = base = folder.name or "images"
    n = 2
    while name in used_names:
        name, n = f"{base}_{n}", n + 1
    used_names.add(name)

    datasets.append(
        {
            "name": name,
            "folder": folder,
            "workflow_root": workflow_root,
            "files": files,
            "config": cfg,
            "config_path": cfg_path,
            "config_source": cfg_source,
            "im_type": ds_im_type,
        }
    )

if not datasets:
    raise FileNotFoundError(
        "No usable datasets found. Check SOURCE_FOLDERS, IM_TYPE, and dataset config files/settings."
    )

pd.set_option("display.max_colwidth", None)
pd.DataFrame(
    [
        {
            "dataset": d["name"],
            "source folder": str(d["folder"]),
            "workflow folder": str(d["workflow_root"]),
            "images": len(d["files"]),
            "im_type": d["im_type"],
            "config": str(d["config_path"]),
            "config_source": d["config_source"],
            "cell_source": d["config"]["CELL_SOURCE"],
            "segment_cells": d["config"]["SEGMENT_CELLS"],
            "segment_nuclei": d["config"]["SEGMENT_NUCLEI"],
        }
        for d in datasets
    ]
)

,dataset,source folder,workflow folder,images,im_type,config,config_source,cell_source,segment_cells,segment_nuclei
0,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test,10,.tiff,/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test/cellpose_config.json,config-file,composite,True,False


---------------------
## **STEP 3: Prepare TIFF inputs for Cellpose**

#### ▶ Run this cell (no user input required)

For each image, this step:
- builds the cell signal (single channel or composite)
- prepares nuclei channel when needed
- normalizes and optionally downsamples XY
- writes TIFF input files and metadata JSON

If one image fails, processing continues for the remaining images.

In [5]:
records = []
for ds in datasets:
    cfg = ds["config"]

    segment_cells = bool(cfg["SEGMENT_CELLS"])
    segment_nuclei = bool(cfg["SEGMENT_NUCLEI"])
    use_nuc_for_cells = bool(cfg["USE_NUCLEI_FOR_CELLS"])
    cell_file_has_nuc = bool(segment_cells and use_nuc_for_cells)
    separate_nuc_file = bool(segment_nuclei and not cell_file_has_nuc)
    need_nuc_channel = bool(cell_file_has_nuc or segment_nuclei)

    xy_downsample = int(cfg["XY_DOWNSAMPLE"])
    pm_ch = int(cfg["PM_CH"])
    nuc_ch = int(cfg["NUC_CH"])

    needed_channels = [nuc_ch] if need_nuc_channel else []
    if segment_cells:
        if cfg["CELL_SOURCE"] == "composite":
            needed_channels += [int(c) for c, w in cfg["COMPOSITE_WEIGHTS"].items() if float(w) > 0]
            if bool(cfg.get("INVERT_PM", False)):
                needed_channels.append(pm_ch)
        else:
            needed_channels.append(pm_ch)

    root = Path(ds["workflow_root"])
    dirs = {k: root / k for k in ("cell_input", "nuclei_input", "metadata", "cell_masks", "nuclei_masks")}
    ds["dirs"] = dirs
    ds["runtime"] = {
        "cell_file_has_nuc": cell_file_has_nuc,
        "separate_nuc_file": separate_nuc_file,
        "need_nuc_channel": need_nuc_channel,
    }

    for key, path_obj in dirs.items():
        if (key == "cell_input" and segment_cells) or (key == "nuclei_input" and separate_nuc_file) or (
            key == "cell_masks" and segment_cells
        ) or (key == "nuclei_masks" and segment_nuclei) or key == "metadata":
            path_obj.mkdir(parents=True, exist_ok=True)

    for src in ds["files"]:
        stem = Path(src).stem
        cell_f = dirs["cell_input"] / f"{stem}.tif"
        nuc_f = dirs["nuclei_input"] / f"{stem}.tif"
        meta_f = dirs["metadata"] / f"{stem}.json"

        expected = [meta_f] + ([cell_f] if segment_cells else []) + ([nuc_f] if separate_nuc_file else [])

        rec = {
            "dataset": ds["name"],
            "workflow_root": str(root),
            "image": stem,
            "status": "",
            "anisotropy": np.nan,
            "prepared shape (Z,Y,X)": "",
            "message": "",
        }

        try:
            if SKIP_EXISTING and all(f.exists() for f in expected):
                info = json.loads(meta_f.read_text())
                rec.update(
                    status="skipped (already prepared)",
                    anisotropy=info.get("anisotropy", np.nan),
                    **{"prepared shape (Z,Y,X)": str(tuple(info.get("prepared_shape_zyx", [])))},
                )
            else:
                img, meta = read_czi_image(src)
                bad = [c for c in sorted(set(needed_channels)) if c >= img.shape[0]]
                if bad:
                    raise ValueError(
                        f"image has {img.shape[0]} channels (0-{img.shape[0] - 1}) but channel(s) {bad} are needed"
                    )

                scale = [float(s) for s in meta["scale"]]
                aniso = scale[0] / (scale[1] * xy_downsample)

                nuc = downsample_zyx(normalize_volume_01(img[nuc_ch]), xy_downsample) if need_nuc_channel else None
                cell = downsample_zyx(build_cell_signal(img, cfg), xy_downsample) if segment_cells else None

                if segment_cells:
                    out = np.stack([cell, nuc], axis=1) if cell_file_has_nuc else cell  # ZCYX or ZYX
                    tifffile.imwrite(cell_f, out.astype(np.float32))
                if separate_nuc_file and nuc is not None:
                    tifffile.imwrite(nuc_f, nuc.astype(np.float32))

                shape = (cell if cell is not None else nuc).shape
                meta_f.write_text(
                    json.dumps(
                        {
                            "source_file": str(src),
                            "scale_zyx": scale,
                            "xy_downsample": xy_downsample,
                            "anisotropy": aniso,
                            "original_shape_zyx": list(img.shape[1:]),
                            "prepared_shape_zyx": list(shape),
                            "cell_file_channels": ["cell signal", "nuclei"] if cell_file_has_nuc else (["cell signal"] if segment_cells else []),
                            "cell_source": cfg["CELL_SOURCE"] if segment_cells else None,
                            "nuclei_channel": nuc_ch if need_nuc_channel else None,
                            "pm_channel": pm_ch,
                            "composite_weights": cfg.get("COMPOSITE_WEIGHTS") if cfg["CELL_SOURCE"] == "composite" else None,
                            "invert_pm": bool(cfg.get("INVERT_PM", False)),
                            "config_file": str(ds["config_path"]) if ds.get("config_path") else None,
                            "config_version": cfg.get("config_version"),
                        },
                        indent=1,
                    )
                )

                rec.update(
                    status="prepared",
                    anisotropy=round(aniso, 4),
                    **{"prepared shape (Z,Y,X)": str(tuple(shape))},
                )
                del img, nuc, cell

        except Exception as e:
            rec.update(status="ERROR", message=f"{type(e).__name__}: {e}")

        records.append(rec)
        print(f"{ds['name']} / {stem}: {rec['status']} {rec['message']}")

prep_table = pd.DataFrame(records)
if not prep_table.empty:
    for ds in datasets:
        ds_table = prep_table[prep_table["dataset"] == ds["name"]]
        ds_csv = Path(ds["workflow_root"]) / "prepared_images.csv"
        ds_table.to_csv(ds_csv, index=False)
        print(f"Wrote dataset summary: {ds_csv}")

if (prep_table["status"] == "ERROR").any():
    print("\nSome images failed (see the 'message' column below). The others were prepared normally.")
prep_table

Attempted file (/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img10_zstack_Unmixing_decon.ome.tiff) load with reader: aicsimageio.readers.bfio_reader.OmeTiledTiffReader failed with error: No module named 'bfio'


Attempted file (/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img11_zstack_Unmixing_decon.ome.tiff) load with reader: aicsimageio.readers.bfio_reader.OmeTiledTiffReader failed with error: No module named 'bfio'


KK-1 / 04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img10_zstack_Unmixing_decon.ome: prepared 


Attempted file (/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img12_zstack_Unmixing_decon.ome.tiff) load with reader: aicsimageio.readers.bfio_reader.OmeTiledTiffReader failed with error: No module named 'bfio'


KK-1 / 04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img11_zstack_Unmixing_decon.ome: prepared 


Attempted file (/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img13_zstack_Unmixing_decon.ome.tiff) load with reader: aicsimageio.readers.bfio_reader.OmeTiledTiffReader failed with error: No module named 'bfio'


... (10 outputs left out)

KK-1 / 04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img6_zstack_Unmixing_decon.ome: prepared 


Attempted file (/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img9_zstack_Unmixing_decon.ome.tiff) load with reader: aicsimageio.readers.bfio_reader.OmeTiledTiffReader failed with error: No module named 'bfio'


KK-1 / 04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img8_zstack_Unmixing_decon.ome: prepared 
KK-1 / 04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img9_zstack_Unmixing_decon.ome: prepared 
Wrote dataset summary: /users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test/prepared_images.csv


,dataset,workflow_root,image,status,anisotropy,"prepared shape (Z,Y,X)",message
0,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img10_zstack_Unmixing_decon.ome,prepared,5.5131,"(21, 1272, 1272)",
1,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img11_zstack_Unmixing_decon.ome,prepared,5.5131,"(22, 1468, 1468)",
2,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img12_zstack_Unmixing_decon.ome,prepared,5.5131,"(22, 1364, 1364)",
3,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img13_zstack_Unmixing_decon.ome,prepared,5.5131,"(16, 1364, 1364)",
4,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img14_zstack_Unmixing_decon.ome,prepared,5.5131,"(18, 956, 956)",
5,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img15_zstack_Unmixing_decon.ome,prepared,5.5131,"(20, 1592, 1592)",
6,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img2_zstack_Unmixing_decon.ome,prepared,5.5131,"(24, 1124, 1124)",
7,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img6_zstack_Unmixing_decon.ome,prepared,5.5131,"(21, 1004, 1004)",
8,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img8_zstack_Unmixing_decon.ome,prepared,5.5131,"(22, 908, 908)",
9,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAMP1_30ng_OxVenSiT_mAppRab11_30ng_PM_lipoLtx4hr_img9_zstack_Unmixing_decon.ome,prepared,5.5131,"(21, 764, 764)",


---------------------
## **STEP 4: Write Cellpose commands**

#### ▶ Run this cell (no user input required)

This step writes **one command file per dataset** using your OS format:
- Windows: `<dataset_workflow_folder>/run_cellpose_batch.bat`
- Linux/macOS: `<dataset_workflow_folder>/run_cellpose_batch.sh`

This keeps multi-dataset runs isolated exactly like separate runs.

In [8]:
good = prep_table[prep_table["status"].isin(["prepared", "skipped (already prepared)"])]
if good.empty:
    raise RuntimeError("No images were prepared successfully, so there are no commands to write. See the table above.")

written_scripts = []
is_windows = os.name == "nt"

for ds in datasets:
    sub = good[good["dataset"] == ds["name"]]
    if sub.empty:
        continue

    cfg = ds["config"]
    rt = ds["runtime"]
    dirs = ds["dirs"]

    segment_cells = bool(cfg["SEGMENT_CELLS"])
    segment_nuclei = bool(cfg["SEGMENT_NUCLEI"])
    cell_file_has_nuc = bool(rt["cell_file_has_nuc"])
    separate_nuc_file = bool(rt["separate_nuc_file"])

    flow_threshold = cfg["FLOW_THRESHOLD"]
    cellprob_threshold = cfg["CELLPROB_THRESHOLD"]
    batch_size = cfg["BATCH_SIZE"]
    use_gpu = bool(cfg.get("USE_GPU", True))

    cell_model = cfg.get("CP_CELL_MODEL_PATH") or "cyto3"
    nuc_model = cfg.get("CP_NUC_MODEL_PATH") or "nuclei"

    groups = sub.groupby(sub["anisotropy"].round(3))

    if is_windows:
        lines = [
            "@echo off",
            "chcp 65001 >nul",
            f"REM Dataset: {ds['name']}",
            f"REM Config: {ds['config_path']}",
            "REM Run this file from a terminal where the environment with Cellpose 3 (cellpose<4) is activated.",
            "REM The masks are saved as <image name>_cp_masks.tif in the cell_masks / nuclei_masks folders.",
            "",
        ]
    else:
        lines = [
            "#!/usr/bin/env bash",
            "set -euo pipefail",
            f"# Dataset: {ds['name']}",
            f"# Config: {ds['config_path']}",
            "# Run this file from a terminal where the environment with Cellpose 3 (cellpose<4) is activated.",
            "# The masks are saved as <image name>_cp_masks.tif in the cell_masks / nuclei_masks folders.",
            "",
        ]

    for name, wanted in (("cells", segment_cells), ("nuclei", segment_nuclei)):
        if not wanted:
            continue

        if name == "cells":
            indir, savedir, model = dirs["cell_input"], dirs["cell_masks"], cell_model
            chan, chan2, axis = (1, 2, 1) if cell_file_has_nuc else (0, 0, None)
            diam, msize = cfg["CELL_DIAMETER"], cfg["MIN_SIZE_CELL"]
        else:
            savedir, model, diam, msize = dirs["nuclei_masks"], nuc_model, cfg["NUC_DIAMETER"], cfg["MIN_SIZE_NUC"]
            if separate_nuc_file:
                indir, chan, chan2, axis = dirs["nuclei_input"], 0, 0, None
            else:
                indir, chan, chan2, axis = dirs["cell_input"], 2, 0, 1

        if len(groups) == 1:
            aniso = float(sub["anisotropy"].iloc[0])
            lines.append(f"REM {name}" if is_windows else f"# {name}")
            lines.append(
                build_cellpose_cli(
                    indir_or_file=Path(indir),
                    savedir=Path(savedir),
                    model=str(model),
                    chan=int(chan),
                    chan2=int(chan2),
                    diameter=int(diam),
                    min_size=int(msize),
                    anisotropy=float(aniso),
                    flow_threshold=float(flow_threshold),
                    cellprob_threshold=float(cellprob_threshold),
                    batch_size=int(batch_size),
                    use_gpu=bool(use_gpu),
                    channel_axis=axis,
                    is_file=False,
                )
            )
        else:
            lines.append(
                f"REM {name}: images have different Z/XY pixel-size ratios, so one command per image"
                if is_windows
                else f"# {name}: images have different Z/XY pixel-size ratios, so one command per image"
            )
            for _, row in sub.iterrows():
                lines.append(
                    build_cellpose_cli(
                        indir_or_file=Path(indir) / f"{row['image']}.tif",
                        savedir=Path(savedir),
                        model=str(model),
                        chan=int(chan),
                        chan2=int(chan2),
                        diameter=int(diam),
                        min_size=int(msize),
                        anisotropy=float(row["anisotropy"]),
                        flow_threshold=float(flow_threshold),
                        cellprob_threshold=float(cellprob_threshold),
                        batch_size=int(batch_size),
                        use_gpu=bool(use_gpu),
                        channel_axis=axis,
                        is_file=True,
                    )
                )
    lines.append("")
    lines.append("echo Done.")

    script_name = "run_cellpose_batch.bat" if is_windows else "run_cellpose_batch.sh"
    script_file = Path(ds["workflow_root"]) / script_name
    newline = "\r\n" if is_windows else "\n"
    script_file.write_text(newline.join(lines) + newline, encoding="utf-8")

    if not is_windows:
        script_file.chmod(0o755)

    written_scripts.append(script_file)

print("Command files saved:")
for p in written_scripts:
    print(f"- {p}")

Command files saved:
- /users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test/run_cellpose_batch.sh


---------------------
## **STEP 5: Run Cellpose from a terminal**

1. Open a terminal.
2. Activate an environment with **Cellpose 3** (`cellpose<4`; Cellpose 4 does not include `cyto3`/`nuclei`).
3. Run each command file path printed in Step 4 (one per dataset).
   - Windows: run `run_cellpose_batch.bat`
   - Linux/macOS: run `run_cellpose_batch.sh`

Notes:
- The first run may download model files if missing.
- Re-running commands overwrites existing mask TIFFs.
- Output folders are created in Step 3; if deleted, rerun Step 3 first.

---------------------
## **STEP 6: Use the results**

- Masks are written as `<image name>_cp_masks.tif` in each dataset workflow folder under `cell_masks/` and `nuclei_masks/`.
- If `XY_DOWNSAMPLE > 1`, masks are smaller than the original image size.
- Per-image shape and scale metadata are in `metadata/<image name>.json`.
- `prepared_images.csv` is written per dataset workflow folder.
- These outputs are designed to stay consistent with `NEW_infer_masks_cellpose.ipynb` settings and config schema.

#### Troubleshooting
| Problem | What to do |
|---|---|
| "No images found" | Check `SOURCE_FOLDERS` and `IM_TYPE`, or enable auto-detect fallback. |
| "image has N channels but channel(s) [..] are needed" | A channel index in settings is larger than channels present in the image. |
| `--savedir ... does not exist` | Re-run Step 3 to recreate output folders. |
| Cellpose cannot find `cyto3` or `nuclei` | Use an environment with `cellpose<4`. |
| Out of GPU memory | Lower `BATCH_SIZE` or raise `XY_DOWNSAMPLE`, then rerun Steps 3-4 with `SKIP_EXISTING = False`. |
| Changed channels/weights/downsample | Set `SKIP_EXISTING = False` so prepared TIFF files are rebuilt. |